# Colab GPU 확인
Colab에서 이 노트북을 업로드하고 GPU 런타임에 연결한 뒤 아래 셀을 실행하세요. 출력된 GPU 이름과 메모리로 기존 Vast 모델을 동일하게 올릴 수 있는지 판단합니다.

이 단계는 하드웨어와 설치된 패키지 버전만 조회합니다. 모델 다운로드, Vast 접속, Google Drive 마운트, 실험 실행을 하지 않습니다. 구독 플랜과 남은 컴퓨팅 단위는 Colab 화면에서 별도로 확인하세요.

다음 단계는 같은 모델·revision·생성 설정의 적재 및 제한된 처리량 시험입니다. 이 조회 성공만으로 모델 적재나 병렬 연동이 완료된 것은 아닙니다.

In [ ]:
import datetime
import importlib.metadata
import json
import platform
import shutil
import subprocess
from pathlib import Path

report = {
    'checked_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'python': platform.python_version(),
    'packages': {},
    'target_model': 'LGAI-EXAONE/EXAONE-4.5-33B-AWQ',
    'target_revision': '31e6a965d0661bbe4a8b895e22a77f8271772ba0',
    'target_context_length': 16384,
    'model_load_tested': False,
    'distributed_execution_enabled': False,
}
for package in ('torch', 'transformers', 'sglang'):
    try:
        report['packages'][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        report['packages'][package] = None

try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free,driver_version',
         '--format=csv,noheader'],
        text=True, capture_output=True, timeout=20, check=False,
    )
    report['gpu_query_exit_code'] = result.returncode
    report['gpu_name_total_free_driver'] = result.stdout.strip().splitlines()
    if result.returncode:
        report['gpu_query_error'] = result.stderr.strip()[:500]
except (OSError, subprocess.TimeoutExpired) as exc:
    report['gpu_query_error'] = type(exc).__name__

memory_file = Path('/proc/meminfo')
if memory_file.exists():
    for line in memory_file.read_text().splitlines():
        if line.startswith(('MemTotal:', 'MemAvailable:')):
            key, value = line.split(':', 1)
            report[key + '_GiB'] = round(int(value.split()[0]) / 1024**2, 2)
report['disk_free_GiB'] = round(shutil.disk_usage(Path.cwd()).free / 1024**3, 2)
print(json.dumps(report, ensure_ascii=False, indent=2))
